In [2]:
import os
import glob
import tiktoken
import numpy as np
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sklearn.manifold import TSNE
import plotly.graph_objects as go
from langchain_openai import AzureOpenAI
from langchain_community.chat_models import AzureChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage

c:\Users\AbhinandanKumar\Rag\env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from dotenv import load_dotenv
from Config.settings import settings

In [10]:
llm=AzureChatOpenAI(
    azure_deployment=settings.AZURE_OPENAI_DEPLOYMENT,
    model_name=settings.AZURE_OPENAI_DEPLOYMENT,
    api_key=settings.OPEN_API_KEY,
    temperature=settings.TEMPERATURE,
    api_version=settings.AZURE_OPENAI_API_VERSION,
    azure_endpoint=settings.AZURE_OPENAI_ENDPOINT
)

C:\Users\AbhinandanKumar\AppData\Local\Temp\ipykernel_40756\2923392192.py:1: LangChainDeprecationWarning: The class `AzureChatOpenAI` was deprecated in LangChain 0.0.10 and will be removed in 1.0. An updated version of the class exists in the `langchain-openai package and should be used instead. To use it run `pip install -U `langchain-openai` and import as `from `langchain_openai import AzureChatOpenAI``.
  llm=AzureChatOpenAI(


In [ ]:
from langchain_community.document_loaders import WebBaseLoader

url = "https://www.zillow.com/tech/using-ai-to-understand-the-complexities-and-pitfalls-of-real-estate-data/"

loader = WebBaseLoader(url)
documents = loader.load()

In [6]:
docs = loader.load()
docs[0]

Document(metadata={'source': 'https://www.zillow.com/tech/using-ai-to-understand-the-complexities-and-pitfalls-of-real-estate-data/', 'title': 'Using AI to Understand the Complexities and Pitfalls of Real Estate Data - Zillow Tech Hub', 'language': 'en-US'}, page_content="\n\n\n\n\n\n\nUsing AI to Understand the Complexities and Pitfalls of Real Estate Data - Zillow Tech Hub\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n\n \n\n\n\n\n\n\n\n\n\n\n\n\n            Have questions about buying, selling or renting during COVID-19? \n            Learn more\n\n\n\n\n\n\n\n\nSkip main navigationSign inJoin HomepageBuy Open Buy sub-menuHomes for saleHomes for sale Foreclosures For sale by owner Open houses New construction Coming soon Recent home sales All homes ResourcesHome Buying Guide Foreclosure center Real estate app Down payment assistance Rent Open Rent sub-menuRental listingsSearch apartments for rent Search houses for rent Search all rental listings Browse all r

In [ ]:
all_text = "\n\n".join(doc.page_content for doc in docs)

In [11]:
# How many tokens in all the documents?

encoding = tiktoken.encoding_for_model(llm.model_name)
tokens = encoding.encode(all_text)
token_count = len(tokens)
print(f"Total tokens for {llm.model_name}: {token_count:,}")

Total tokens for gpt-4o: 5,191


In [12]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = text_splitter.split_documents(docs)
# split_documents() method expects  list[Document] and output should be list[Document]
# chunks = text_splitter.split_text(all_text)
# split_text()  expects a string and output will be list of strings
print(f"Divided into {len(chunks)} chunks")
print(f"First chunk:\n\n{chunks[0]}")

Divided into 42 chunks
First chunk:

page_content='Using AI to Understand the Complexities and Pitfalls of Real Estate Data - Zillow Tech Hub




































 












            Have questions about buying, selling or renting during COVID-19? 
            Learn more' metadata={'source': 'https://www.zillow.com/tech/using-ai-to-understand-the-complexities-and-pitfalls-of-real-estate-data/', 'title': 'Using AI to Understand the Complexities and Pitfalls of Real Estate Data - Zillow Tech Hub', 'language': 'en-US'}


In [13]:
from langchain_huggingface import HuggingFaceEmbeddings

In [14]:
db_name = "vector_db1"

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

if os.path.exists(db_name):
    Chroma(persist_directory=db_name, embedding_function=embeddings).delete_collection()

vectorstore = Chroma.from_documents(documents=chunks, embedding=embeddings, persist_directory=db_name)
print(f"Vectorstore created with {vectorstore._collection.count()} documents")

Vectorstore created with 42 documents


In [15]:
# Let's investigate the vectors

collection = vectorstore._collection
count = collection.count()

sample_embedding = collection.get(limit=1, include=["embeddings"])["embeddings"][0]
dimensions = len(sample_embedding)
print(f"There are {count:,} vectors with {dimensions:,} dimensions in the vector store")

There are 42 vectors with 384 dimensions in the vector store


In [16]:
# Get all metadata dictionaries
metadatas = [chunk.metadata for chunk in chunks]

# Collect all unique keys
all_keys = set()
for metadata in metadatas:
    all_keys.update(metadata.keys())

print("All metadata keys found in chunks:")
for key in sorted(all_keys):
    print("-", key)

All metadata keys found in chunks:
- language
- source
- title


In [17]:
# Prework

result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']

# Use 'source' instead of 'doc_type'
doc_types = [metadata['source'] for metadata in metadatas]

# Map unique sources to colors
unique_sources = list(set(doc_types))
color_palette = ['blue', 'green', 'red', 'orange', 'purple', 'brown', 'pink', 'cyan']
colors_map = {src: color_palette[i % len(color_palette)] for i, src in enumerate(unique_sources)}

colors = [colors_map[src] for src in doc_types]

In [18]:
# We humans find it easier to visalize things in 2D!
# Reduce the dimensionality of the vectors to 2D using t-SNE
# (t-distributed stochastic neighbor embedding)

tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.9),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(title='2D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x',yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [19]:
tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=10, b=10, l=10, t=40)
)

fig.show()

In [20]:
vectorstore = Chroma(persist_directory=db_name, embedding_function=embeddings)

In [21]:
retriever = vectorstore.as_retriever()

In [22]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the Finance Knowledge.
You are chatting with a user who has almost zero knowledge in Finance .
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [23]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [24]:
from IPython.display import display, Markdown

display(Markdown(answer_question("can you please give me the summary ? ",history=[])))


Sure! Here's a summary of the context provided:

The analysis explores techniques to understand the semantic differences between two groups of listing descriptions (majority non-Hispanic white and majority Black neighborhoods). It uses progressively complex methods, including topic modeling with the LDA (Latent Dirichlet Allocation) model. A sample of 10,000 descriptions from each group is analyzed to extract 15 topics. Tools like pyLDAvis are used to visualize and interactively explore these topics, showing their relationships, prominence, and word distributions.

The topic modeling approach addresses limitations of simpler methods, such as key phrase-based analysis, which can be overly simplistic and require manual interpretation. Topic modeling provides a more automated and nuanced understanding of semantic themes, reducing subjectivity and effort in analysis.

In [25]:
display(Markdown(answer_question("isn't this is related to ecommerce business ? ",history=[])))


Not exactly! The context provided is primarily related to real estate, not e-commerce. It discusses themes in property listings, particularly in majority Black locations (Group B), and mentions terms like "investment opportunity," "first-time home buyer," and "home renovations." It also touches on the use of AI and machine learning in real estate, such as Zillow's efforts to enhance user experience and data analysis.

While real estate involves buying and selling, which might seem similar to e-commerce, it is a distinct industry focused on property transactions rather than general goods or services. Let me know if you'd like further clarification!

In [26]:
display(Markdown(answer_question("Basically you have knowledge for the Real Estate right ? If yes can you please what are the best question i can ask you i am a newbie in Real estate business ? ",history=[])))


Yes, I can help you with real estate-related questions! Since you're new to the real estate business, here are some great beginner-friendly questions you can ask to build your knowledge:

### General Real Estate Basics:
1. What is real estate, and how does it work?
2. What are the different types of real estate (residential, commercial, industrial, etc.)?
3. What are the key steps to buying or selling a property?
4. What is the role of a real estate agent, and do I need one?
5. What is the difference between renting and owning a property?

### Investing in Real Estate:
6. How do I start investing in real estate as a beginner?
7. What are the risks and rewards of real estate investing?
8. What is the difference between flipping houses and renting properties?
9. What is a real estate investment trust (REIT)?
10. How do I calculate the return on investment (ROI) for a property?

### Financing and Loans:
11. What is a mortgage, and how does it work?
12. What is the difference between fixed-rate and adjustable-rate mortgages?
13. How much down payment is typically required to buy a property?
14. What is a pre-approval for a mortgage, and why is it important?
15. How do interest rates affect real estate prices?

### Real Estate Market Trends:
16. How do I analyze the real estate market in my area?
17. What factors influence property values?
18. What is the impact of location on real estate prices?
19. How do I identify a buyer's market vs. a seller's market?
20. What are the current trends in the real estate market?

### Legal and Technical Aspects:
21. What is a property deed, and why is it important?
22. What is the process of closing a real estate deal?
23. What are property taxes, and how are they calculated?
24. What is zoning, and how does it affect real estate?
25. What are the common pitfalls to avoid when buying or selling a property?

### Technology in Real Estate:
26. How is AI being used in real estate?
27. What are the benefits of using online platforms like Zillow for buying or selling properties?
28. How can I use data to make better real estate decisions?
29. What is virtual home staging, and how does it work?
30. How can technology help me market my property effectively?

Feel free to ask any of these questions, or let me know if you have something specific in mind!

In [27]:
display(Markdown(answer_question(" what is zillow? ",history=[])))


Zillow is a popular online real estate platform that helps people buy, sell, rent, and finance homes. It provides tools and resources for users to search for properties, estimate home values (using its "Zestimate" feature), and connect with real estate professionals. Zillow also offers mortgage lending services through Zillow Home Loans, LLC, and provides insights into the real estate market using AI-powered tools like personalized home recommendations and virtual tours. Essentially, it's a one-stop shop for navigating the real estate market!

In [28]:
display(Markdown(answer_question("what is the documents all about and what type of problem it solves in your knowledge base? ",history=[])))


The document is about using AI models, specifically Latent Dirichlet Allocation (LDA) and other tools, to analyze unstructured textual data, such as real-estate listing descriptions. It explains how these models can identify patterns, topics, and differences in text data between two groups of documents. In this case, the analysis focuses on comparing real-estate listings in areas with different racial demographics (majority non-Hispanic white vs. majority Black groups).

### What Problem Does It Solve?
The document addresses the challenge of analyzing unstructured text data to uncover meaningful insights. Specifically, it solves the problem of understanding whether text data can act as a proxy for protected classes (e.g., race or ethnicity) and whether there are notable differences in the way homes are described in different demographic areas. This is important for ensuring fairness and avoiding bias in AI systems that process textual data.

### Why Is This Useful?
1. **Understanding Bias**: It helps identify potential biases in textual descriptions, which could influence decision-making processes in real estate or other industries.
2. **Automating Analysis**: By using AI models like LDA, the process of analyzing large amounts of text data becomes automated and scalable.
3. **Improving Interpretability**: Tools like pyLDAvis and LLMs make it easier to interpret the results of topic modeling, allowing users to explore and understand the data interactively.
4. **Fairness in AI**: It contributes to ensuring that AI systems do not unintentionally discriminate based on textual proxies for protected classes.

In summary, the document showcases how AI can be leveraged to analyze unstructured text data, uncover differences between groups, and ensure fairness in applications like real estate or other industries dealing with textual data.

In [29]:
display(Markdown(answer_question("What are the risks and rewards of real estate investing?",history=[])))


Real estate investing can be a great way to build wealth, but it also comes with its own set of risks and rewards. Let me break it down for you in simple terms:

### **Rewards of Real Estate Investing**
1. **Steady Income**: If you invest in rental properties, you can earn a steady stream of income from tenants paying rent.
2. **Appreciation**: Over time, real estate properties often increase in value, allowing you to sell them for a profit in the future.
3. **Tax Benefits**: Real estate investors can take advantage of tax deductions, such as mortgage interest, property taxes, and depreciation.
4. **Diversification**: Adding real estate to your investment portfolio can help spread risk, as it often behaves differently than stocks or bonds.
5. **Control Over Investment**: Unlike stocks, you have more control over your real estate investment. You can make improvements to increase its value or manage it to maximize income.
6. **Hedge Against Inflation**: Real estate values and rents often rise with inflation, protecting your purchasing power.

---

### **Risks of Real Estate Investing**
1. **High Upfront Costs**: Buying property requires a significant amount of money for the down payment, closing costs, and potential renovations.
2. **Market Fluctuations**: Real estate markets can go up and down, and property values may decrease during economic downturns.
3. **Illiquidity**: Real estate is not as easy to sell as stocks or bonds. It can take time to find a buyer and close a deal.
4. **Ongoing Expenses**: Owning property comes with costs like maintenance, property taxes, insurance, and possibly mortgage payments.
5. **Tenant Issues**: If you rent out your property, you may face challenges like late payments, property damage, or vacancies.
6. **Economic Risks**: Broader economic factors, such as interest rate hikes or a recession, can impact property values and rental demand.
7. **Legal and Regulatory Risks**: Real estate is subject to local laws and regulations, which can change and affect your investment.

---

### **How to Mitigate Risks**
- **Do Your Research**: Understand the local market and property values before investing.
- **Diversify**: Don’t put all your money into one property or one type of real estate.
- **Have a Financial Cushion**: Keep savings for unexpected expenses like repairs or vacancies.
- **Work with Professionals**: Hire real estate agents, property managers, or financial advisors to guide you.
- **Understand the Laws**: Be aware of landlord-tenant laws and zoning regulations in your area.

Real estate investing can be rewarding, but it’s important to weigh the risks and rewards carefully and make informed decisions. Let me know if you’d like to dive deeper into any of these points!